In [1]:
!pip -q install chromadb langchain langchain_community huggingface langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 

# Using Huggingface embedding model for creating text embedding

In [2]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-small-en-v1.5"

embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'}
)

print(f"Embedding model loaded: {model_name}")

/tmp/ipykernel_935/2583931005.py:5: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: BAAI/bge-small-en-v1.5


In [3]:
texts = [
    "This is a document about machine learning.",
]

doc_embeddings = embeddings.embed_documents(texts)



In [4]:
doc_embeddings[0][:3]

[-0.017054904252290726, 0.01815015636384487, 0.0047840699553489685]

In [5]:
doc_embeddings[0][:5]

[-0.017054904252290726,
 0.01815015636384487,
 0.0047840699553489685,
 0.006994710303843021,
 0.06046576797962189]

In [6]:
query_text = "What is Machine learning?"
query_embedding=embeddings.embed_query(query_text)
query_embedding[0]

0.005669789854437113

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
db = Chroma.from_texts(texts, embeddings)

In [9]:
docs = db.similarity_search(query_text)
print(docs[0].page_content)

This is a document about machine learning.


In [10]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=4,chunk_overlap=2)

In [11]:
text=text_splitter.split_text(texts[0])
text

['This',
 'is',
 'a',
 'doc',
 'ocum',
 'umen',
 'ent',
 'abo',
 'bout',
 'mac',
 'achi',
 'hine',
 'lea',
 'earn',
 'rnin',
 'ing.']

In [12]:
len(text)

16

In [13]:
text[1]

'is'

In [38]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sbhatti/news-articles-corpus")

print("Path to dataset files:", path)

100%|██████████| 1.60G/1.60G [00:20<00:00, 83.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sbhatti/news-articles-corpus/versions/1


In [40]:
import pandas as pd

In [45]:
import os

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if csv_files:
    csv_file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_file_path)
    print(df.head())
else:
    print(f"No CSV files found in the directory: {path}")

No CSV files found in the directory: /root/.cache/kagglehub/datasets/sbhatti/news-articles-corpus/versions/1


In [14]:
persist_dir="db"


embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'}
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
embeddings.model_name

'BAAI/bge-small-en-v1.5'

In [16]:
vector_db=Chroma.from_texts(
    texts,
    embeddings,
    persist_directory=persist_dir,
    collection_name="my_collection"
)

In [17]:
vector_db.persist()
vector_db=None

/tmp/ipykernel_935/2118368330.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


In [35]:
vector_db=Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
    collection_name="my_collection")

In [36]:
retriever=vector_db.as_retriever()

In [37]:
query="What is machine learning?"
vector_db.similarity_search(query)

[Document(metadata={}, page_content='This is a document about machine learning.')]